# Module 7, Section 4: LLM Gateway and Governance

Sections 1 to 3 found problems after they happened. This section is about controls that stop some of them from happening at all. The LangSmith docs describe this part of governance as what agents do at runtime: spend and data protection policies on model calls, block and approval policies on tool calls, sandboxed execution for agent code, and human approval before sensitive actions.

Module 4 Section 2 added `PIIMiddleware` to the deep support agent and ended with this warning:

> ⚠️ **Traces are a separate control.** `PIIMiddleware` rewrites the conversation state, but the raw run input and each span's raw output are still recorded in LangSmith. To keep PII out of traces, add LangSmith's trace masking (hide or anonymize inputs and outputs) as well.

This section measures that gap and closes it.

In this section:
1. An email address that a customer typed, still in yesterday's traces
2. Three controls for personal data, and where each one runs
3. Proof of what leaves the process, with masking off and on
4. The LLM Gateway, and how a model call finds it
5. What a policy block looks like to the agent
6. Which control stops which of the day's five incidents
7. Who did what: traces, the audit log, roles and Custom Apps
8. Where to go next

## Setup

Everything in this section runs with no key and calls no model. The agents in §3 use a scripted model, and their traces go to a recorder on this machine. One optional cell in §7 calls the LLM Gateway for real; it is off unless you turn it on.

In [ ]:
import json, os, sys, warnings
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT))
# Load .env before any workshop import: config.py picks the models from it at import time
load_dotenv(REPO_ROOT / ".env")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*is deprecated and will be removed after", category=DeprecationWarning)

from utils.langsmith_links import setup_langsmith
from utils.trace_seeding import describe, load_seed, print_trace
from utils.trace_upload import resolve_destination

LANGSMITH_AVAILABLE = setup_langsmith()            # one probe at most, one printed line
DEST = resolve_destination(LANGSMITH_AVAILABLE)    # project name, project id, navigation sentence
DATA = REPO_ROOT / "workshop_modules" / "module_7" / "data"
seed = load_seed()                                 # timestamps shifted so the newest run is 5 minutes old

## 1. An email address in yesterday's traces

In one conversation of the seeded day, a customer typed an email address into the chat. `PIIMiddleware` was on, as it is by default in the deep support agent. The next cell searches the inputs and outputs of every root run for an email address.

With a key, and with the day uploaded in Section 1, you can open the same trace in LangSmith:

> In LangSmith, open **Tracing**. If the control at the top left shows your workspace name (an agent-based workspace), select the agent **TechHub support** in the agent switcher in the top bar, then set the environment control to **Production**. If it shows the LangSmith logo (a project-based workspace), open the tracing project `techhub-deep-agent-production` (or the name you set in `M7_PROJECT`). Set the time range to at least the last 2 days.

In [ ]:
import re
from utils.monitoring_checks import check
from utils.trace_seeding import EMAIL_PATTERN, load_labels, question_text, root_runs, trace_key

EMAIL = re.compile(EMAIL_PATTERN)
roots = root_runs(seed)
in_inputs = [run for run in roots if EMAIL.search(json.dumps(run["inputs"]))]
in_outputs = [run for run in roots if EMAIL.search(json.dumps(run["outputs"] or {}))]
leaked = {trace_key(run) for run in in_inputs + in_outputs}
print(f"{len(leaked)} root run contains a raw email address: {len(in_inputs)} in its inputs, {len(in_outputs)} in its outputs.")

labels = load_labels()
topic = {key: c["topic"] for c in labels["conversations"] for key in c["trace_keys"]}
for run in in_inputs:
    key = trace_key(run)
    saved = run["outputs"]["messages"][0]["content"]  # the same customer message, as the conversation state kept it
    print()
    print(f"{key} (topic {topic[key]}, app_version {run['extra']['metadata']['app_version']})")
    print("  inputs.messages[0].content: ", question_text(run))
    print("  outputs.messages[0].content:", saved)
    print(f'  In the LangSmith search bar: and(eq(metadata_key, "seed_trace_key"), eq(metadata_value, "{key}"))')
print()

ok, message = check("s4_leak", {
    "root_runs_with_raw_email": len(leaked),
    "in_inputs": len(in_inputs),
    "in_outputs": len(in_outputs),
})
print(message)

## 2. Three controls, three places

The middleware did its job: the model and the saved conversation only ever saw `[REDACTED_EMAIL]`. The root run's inputs are recorded before the middleware runs, so the trace still holds what the customer typed. Three controls deal with personal data, and each one protects a different copy of it.

| | `PIIMiddleware` (LangChain) | Gateway data policy (LangSmith) | Client-side trace masking (LangSmith SDK) |
|---|---|---|---|
| Where it runs | In your agent process, around each model call | In the LLM Gateway, between your process and the model provider | In your process, just before a trace is uploaded |
| Who sets it up | The developer, in code | An organization admin, in the UI, with no code change | The developer, with `Client(anonymizer=...)`, `hide_inputs` or an environment variable |
| What it protects | What the model sees, and the saved conversation | What the model provider sees, and what the `gateway` trace stores | What LangSmith stores in your application's traces |
| Does the model see the original? | No | No: it gets placeholders, and the Gateway restores the values in the response | Yes |
| Does your own LangSmith trace keep the original? | Yes: the root input, the middleware's own spans and the raw model output | Yes: your application receives the restored values | No |
| Gaps | One PII type per middleware instance. Subagents do not inherit it | Scans requests, not responses. Skips system prompts. Needs a PII redaction entitlement that is off by default | Run names, tags and attachments are not masked |

A fourth control exists on the Enterprise plan: the permissions `runs:read-inputs` and `runs:read-outputs`, and an access policy that hides inputs and outputs in projects with a given tag. The data is stored, and selected users cannot read it.

## 3. Prove what leaves the process

The next cells trace a small agent into a **capture client**: a LangSmith client whose HTTP layer records each upload and answers it on this machine. Nothing leaves the machine, and no key is needed. `utils/offline_capture.py` has the code.

The agent is built with `create_agent`, the same function the deep support agent is built on, with no tools and a scripted model that returns one prepared reply. The customer's message holds an email address and an order id. The order id is the control string: it is not personal data, so masking must leave it alone.

Run 1 has `PIIMiddleware` for email on input and output, as in the deep support agent, and no masking in the client.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langsmith import tracing_context
from utils.monitoring_checks import MASKING_CONTROL, MASKING_EMAIL, MASKING_QUESTION, MASKING_REPLY
from utils.offline_capture import count_payloads_containing, offline_client, scripted_model, uploaded_payloads


def email_middleware():
    return PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True)


def run_once(middleware, **client_kwargs):
    """Run the agent once, traced into a capture client. Returns the final state and the capture."""
    agent = create_agent(scripted_model([MASKING_REPLY]), tools=[], middleware=middleware)
    client, capture = offline_client(**client_kwargs)
    with tracing_context(enabled=True, client=client):
        state = agent.invoke({"messages": [{"role": "user", "content": MASKING_QUESTION}]})
    client.flush()
    return state, capture


def leak_line(label, capture):
    total = len(uploaded_payloads(capture))
    return f"{label}: {count_payloads_containing(capture, MASKING_EMAIL)} of {total} uploaded payloads contain {MASKING_EMAIL}."


state, run1 = run_once([email_middleware()])
print("Final conversation state:")
for message in state["messages"]:
    print(f"  {message.type}: {message.content}")
print()
print(leak_line("Run 1, middleware only", run1))
where = sorted({(payload.get("name"), part) for payload in uploaded_payloads(run1)
                for part in ("inputs", "outputs") if MASKING_EMAIL in json.dumps(payload.get(part))})
for name, part in where:
    print(f"  {name}: {part}")

The model and the conversation state saw `[REDACTED_EMAIL]`, yet most uploaded payloads still hold the address. The list shows where: the root run's inputs (`LangGraph`), the inputs of the middleware's own `before_model` and `after_model` spans, and the raw model output before the middleware rewrote it. Each run is uploaded twice, once when it starts and once when it ends, so one run can count twice. On our runs this was 8 of 10. The exact count depends on the installed LangChain version.

An **anonymizer** is a function that the LangSmith client applies to the inputs, outputs, metadata and error of every run just before upload. `langsmith.anonymizer.create_anonymizer` builds one from a list of rules of the form `{"pattern": <regular expression>, "replace": <text>}`.

> 💡 **Try it first:** write `build_anonymizer()` in `workshop_modules/module_7/exercises/masking.py` so that no uploaded payload contains an email address and the order id survives. Check it from the repository root with `uv run python workshop_modules/module_7/exercises/check.py s4_masking`. The next cell shows the reference version.

In [ ]:
from langsmith.anonymizer import create_anonymizer


def build_anonymizer():
    """Replace every email address with <email> and leave all other text alone."""
    email = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
    return create_anonymizer([{"pattern": email, "replace": "<email>"}])


# Run 2: the same middleware, and a client that masks before upload.
state2, run2 = run_once([email_middleware()], anonymizer=build_anonymizer())
print(leak_line("Run 2, middleware and anonymizer", run2))
control_before = count_payloads_containing(run1, MASKING_CONTROL)
control_after = count_payloads_containing(run2, MASKING_CONTROL)
print(f"  The order id {MASKING_CONTROL} is in {control_before} payloads in run 1 and {control_after} in run 2: unchanged.")

# Run 3: the anonymizer alone, no middleware.
state3, run3 = run_once([], anonymizer=build_anonymizer())
print(leak_line("Run 3, anonymizer only", run3))
print("  What the model received:", state3["messages"][0].content)
print()

ok, message = check("s4_masking", build_anonymizer)
print(message)

In [ ]:
from langchain_core.runnables import RunnableLambda
from utils.offline_capture import hide_inputs_in_subprocess

# Trap 1: LANGSMITH_HIDE_INPUTS, one new Python process per spelling, because the SDK caches the first value it reads.
hidden = {value: hide_inputs_in_subprocess(value)["uploaded_inputs"] == {} for value in ("true", "True", "1")}

# Trap 2: tracing_context(enabled="local") still uploads the runs of a LangChain runnable.
probe = RunnableLambda(lambda inputs: {"answer": "It shipped."}, name="probe")
uploads = {}
for setting in ("local", False):
    client, capture = offline_client()
    with tracing_context(enabled=setting, client=client):
        probe.invoke({"question": "Where is order ORD-2025-0050?"})
    client.flush()
    uploads[setting] = len(uploaded_payloads(capture))

# Trap 3: calling an anonymizer on your own dictionary changes that dictionary.
note = {"note": f"Copy {MASKING_EMAIL} on order {MASKING_CONTROL}."}
result = build_anonymizer()(note)

rows = [
    ("LANGSMITH_HIDE_INPUTS=true, =True, =1", "Inputs hidden for all three",
     "Hidden for " + ", ".join(v for v, h in hidden.items() if h) + "; not hidden for " + ", ".join(v for v, h in hidden.items() if not h)),
    ('tracing_context(enabled="local")', "Nothing is uploaded",
     f'{uploads["local"]} payloads uploaded ({uploads[False]} with enabled=False)'),
    ("anonymizer(my_dict)", "Returns a masked copy",
     f"Returns the same object: {result is note}; my_dict is now {note}"),
]
print(f"{'Setting':<40} {'What you expect':<28} What happens")
for setting, expected, happened in rows:
    print(f"{setting:<40} {expected:<28} {happened}")

The three traps:

- The SDK compares the variable with the exact string `true`. `True` and `1` hide nothing and raise no warning.
- `"local"` keeps a `@traceable` function on the machine, but LangChain runnables still attach their tracer and upload. With a real key, an agent wrapped in `tracing_context(enabled="local")` sends every run. Use `enabled=False` to trace nothing, or an anonymizer to trace with masking.
- Inside the client the anonymizer works on a copy, so this is harmless there. To try an anonymizer on your own data, pass a copy.

## 4. The LLM Gateway: one control layer

The **LLM Gateway** is a LangSmith endpoint in front of the model providers. Your application sends a LangSmith API key, and LangSmith holds the provider keys as workspace secrets. The Gateway is in beta, so labels and behaviour can still change.

For each request, the Gateway:

1. Authenticates the caller from the LangSmith API key and checks the `gateway:invoke` permission.
2. Resolves the route from the endpoint and the model ID, including fallback models.
3. Loads the provider credential (a workspace provider key, or Gateway Credits).
4. Applies the policies: spend, rate and model access policies can block the request, and a data policy redacts it.
5. Translates the request if the provider uses another API format.
6. Sends the request to the provider.
7. Translates the response back and restores redaction placeholders where it applies.
8. Returns the response and records usage, cost, the policy decisions and a trace in the project `gateway`.

LangChain uses direct model access, which skips steps 5 and 7.

There are three routes to the Gateway, and they name models differently:

| Route | What you set | Base URL | Model name |
|---|---|---|---|
| A. LangChain and Deep Agents switch | `LANGSMITH_GATEWAY=true` and `LANGSMITH_API_KEY` (or `LANGSMITH_GATEWAY_API_KEY`) | `https://gateway.smith.langchain.com/openai/v1` for `ChatOpenAI`, `https://gateway.smith.langchain.com/anthropic` for `ChatAnthropic` | Native: `openai:gpt-5.4` |
| B. Standard endpoint, for any OpenAI- or Anthropic-compatible client | The base URL, with the LangSmith key as the client's API key | `https://gateway.smith.langchain.com/v1` | `<provider>/<model>`, such as `openai/gpt-5.4-mini` |
| C. The `langsmith:` provider of `init_chat_model` | `init_chat_model("langsmith:openai/gpt-5.4")` | `https://gateway.smith.langchain.com/v1` | `<provider>/<model>` |

This repo uses route A in `config.py` and route C in `managed_agent/model.py`. The next cell builds two models under five environments and prints where each would send its calls. It never calls them.

In [ ]:
from unittest import mock
from langchain.chat_models import init_chat_model

DUMMY = {  # placeholder values, so the cell can tell which variable a model took its key from
    "LANGSMITH_API_KEY": "dummy-langsmith-key",
    "LANGSMITH_GATEWAY_API_KEY": "dummy-gateway-key",
    "OPENAI_API_KEY": "dummy-openai-key",
    "ANTHROPIC_API_KEY": "dummy-anthropic-key",
}
ENVIRONMENTS = {
    "LANGSMITH_GATEWAY=true, LANGSMITH_API_KEY": {"LANGSMITH_GATEWAY": "true", "LANGSMITH_API_KEY": DUMMY["LANGSMITH_API_KEY"]},
    "... plus LANGSMITH_GATEWAY_API_KEY": {"LANGSMITH_GATEWAY": "true", "LANGSMITH_API_KEY": DUMMY["LANGSMITH_API_KEY"],
                                           "LANGSMITH_GATEWAY_API_KEY": DUMMY["LANGSMITH_GATEWAY_API_KEY"]},
    "LANGSMITH_GATEWAY=true and provider keys": {"LANGSMITH_GATEWAY": "true", "LANGSMITH_API_KEY": DUMMY["LANGSMITH_API_KEY"],
                                                 "OPENAI_API_KEY": DUMMY["OPENAI_API_KEY"], "ANTHROPIC_API_KEY": DUMMY["ANTHROPIC_API_KEY"]},
    "LANGSMITH_GATEWAY=<EU host>": {"LANGSMITH_GATEWAY": "https://eu.gateway.smith.langchain.com", "LANGSMITH_API_KEY": DUMMY["LANGSMITH_API_KEY"]},
    "No LANGSMITH_GATEWAY, provider keys": {"OPENAI_API_KEY": DUMMY["OPENAI_API_KEY"], "ANTHROPIC_API_KEY": DUMMY["ANTHROPIC_API_KEY"]},
}


def key_source(secret):
    value = secret.get_secret_value() if secret else None
    return next((name for name, dummy in DUMMY.items() if dummy == value), "none")


routes = {}
for label, environment in ENVIRONMENTS.items():
    with mock.patch.dict(os.environ, environment, clear=True):  # only these variables, and only inside this block
        openai_model = init_chat_model("openai:gpt-5.4")
        anthropic_model = init_chat_model("anthropic:claude-haiku-4-5")
    routes[label] = str(openai_model.root_client.base_url).rstrip("/")
    print(label)
    print(f"  openai:gpt-5.4              -> {routes[label]}  (key from {key_source(openai_model.openai_api_key)})")
    print(f"  anthropic:claude-haiku-4-5  -> {anthropic_model.anthropic_api_url.rstrip('/')}  (key from {key_source(anthropic_model.anthropic_api_key)})")
print()

ok, message = check("s4_routes", routes["LANGSMITH_GATEWAY=true and provider keys"])
print(message)

With `LANGSMITH_GATEWAY=true` set, calls go to the Gateway with the Gateway key even when provider keys are set too. The repo README says "provider keys win if several are set", and `config.py` then reports the credential source as `environment`. For this case both are wrong: to know where a model sends its calls, read the base URL of the model you built.

## 5. What a policy block looks like

There are five policy types. All are created on tabs of the **LLM Gateway** page, and creating any of them needs the permission `organization:gateway:manage`, which Organization Admins and Organization Operators have. That is why the facilitator creates policies and you read the blocks.

| Policy | Where it is created | Scopes | When it blocks | What the caller gets |
|---|---|---|---|---|
| Spend limit | **Cost Controls** | Organization, workspace, user or API key | Recorded spend in the window (month, week, day or hour) reaches the cap | `402` |
| Rate limit | **Rate limiting** | Same | Requests or tokens in a rolling minute or hour exceed the limit | `429` with a `Retry-After` header |
| Model access | **Model Access** | Same | The provider or model is not on the list | `403`, and the model is missing from `GET /v1/models` |
| Data policy | **Data Policy** | Same | Never, except when a scan times out and **On timeout** is **Block request** | Normally nothing: the request is redacted on the way to the provider |
| Header-based | A condition on one of the policies above | A value of an `X-Gateway-*` request header | As that policy | As that policy |

Every block body contains `request blocked by gateway policies: <policy name>`, and the blocked request is still traced in the project `gateway`.

> 💡 **Try it first:** the next cell replays three saved Gateway responses through `ChatOpenAI`. Before you run it, say which policy type each status code comes from, and which of the three the agent's retry middleware retries.

In [ ]:
import ast
from utils.offline_capture import gateway_replay

# What each status means, from the policy table above (LangSmith docs).
POLICY_BY_STATUS = {402: "spend_limit", 429: "rate_limit", 403: "model_access"}

blocks = {}
for kind in ("spend", "rate", "model_access"):
    try:
        gateway_replay(kind).invoke("ping")  # a ChatOpenAI whose HTTP layer returns the saved response; nothing is sent
    except Exception as exc:  # noqa: BLE001 - every replay ends in the exception LangChain raises for that status
        policy = re.search(r'request blocked by gateway policies: [^"]+', exc.response.text).group(0)
        retry_after = exc.response.headers.get("Retry-After")
        print(f"{exc.status_code}  {type(exc).__name__:<22} Retry-After: {retry_after or '-':<3} {policy}")
        blocks[str(exc.status_code)] = {"policy_type": POLICY_BY_STATUS[exc.status_code], "exception": type(exc).__name__}
print()

# The line of agents/middleware/fault_tolerance.py that decides what ModelRetryMiddleware retries (copied when the notebook was built).
TRANSIENT_LINE = '_TRANSIENT_STATUSES = {408, 409, 429}'
transient = ast.literal_eval(TRANSIENT_LINE.split("=", 1)[1].strip())
print("agents/middleware/fault_tolerance.py:", TRANSIENT_LINE)
retried = sorted(int(status) for status in blocks if int(status) in transient or int(status) >= 500)
print("Of the three blocks, the agent retries:", ", ".join(str(status) for status in retried))
print()

ok, message = check("s4_blocks", {**blocks, "retried_statuses": retried})
print(message)

### A spend cap is not a pre-check

The Gateway records a request's cost after the request completes. The request that crosses the cap still succeeds, and only later requests are blocked, so spend in a window can end up above the cap.

What the TechHub deep agent does on each block, from the source of `agents/middleware/fault_tolerance.py` and the OpenAI SDK (not run against a live Gateway, so UNVERIFIED end to end):

- `429`: the OpenAI SDK retries it twice by default before it raises. Then `ModelRetryMiddleware` retries it (408, 409, 429 and every 5xx status count as transient), and after that `ModelFallbackMiddleware` tries the fallback models.
- `402` and `403`: no retry. The agent goes straight to the fallback models.
- Fallback models called with the same key or in the same workspace fall under the same policy, so they are blocked too, and the run ends with the error.

A rate-limit block never reaches the provider, so it does not start a Gateway fallback chain either, even though `429` is one of the default fallback triggers.

A policy scoped to an API key caps the whole agent. To cap one end customer, the backend sends a header such as `X-Gateway-Customer-Id` after it has authenticated the customer, and a spend limit is separated by that header. The Gateway trusts these headers, so only your backend may set them.

The next cell (120-minute path) estimates who spent what in the seeded day and plays the day's model calls against a daily cap.

In [ ]:
from statistics import median
from utils.monitoring_local import cost_of, spend_by
from utils.trace_seeding import ILLUSTRATIVE_PRICES_PER_1M, runs_by_trace

PRICES = dict(ILLUSTRATIVE_PRICES_PER_1M)  # dollars per 1 million tokens; round numbers invented for this lab, not a price list
print("Illustrative prices per 1M tokens:", ", ".join(f"{model} {p['input']:g} in / {p['output']:g} out" for model, p in PRICES.items()))

spend = spend_by(seed, "customer_id", PRICES)
total = sum(spend.values())
print(f"Estimated spend for the day: ${total:.2f}")
print("Top customers:", ", ".join(f"{customer} ${amount:.2f}" for customer, amount in list(spend.items())[:3]))

trace_cost = {key: sum(cost_of(run, PRICES) for run in runs) for key, runs in runs_by_trace(seed).items()}
per_conversation = {}
for conversation in labels["conversations"]:
    per_conversation.setdefault(conversation["subcategory"], []).append(sum(trace_cost[key] for key in conversation["trace_keys"]))
by_median = sorted(per_conversation.items(), key=lambda item: -median(item[1]))
print("Median cost per conversation:", ", ".join(f"{name} ${median(costs):.3f}" for name, costs in by_median[:3]))
print(f"Top spender: {next(iter(spend))}. Costliest subcategory: {by_median[0][0]}.")
print()

# Every model call is one Gateway request. This sketch puts the whole day in one daily window and
# records each request's cost before the next request starts.
requests = sorted((run for run in seed["runs"] if run["run_type"] == "llm"), key=lambda run: run["end_time"])
CAP = round(total / 2, 2)  # a daily spend limit on the agent's API key, set at half of the day's spend
recorded = 0.0
for number, run in enumerate(requests, start=1):
    cost = cost_of(run, PRICES)
    if recorded + cost >= CAP:
        break
    recorded += cost
print(f"Daily cap on the agent's API key: ${CAP:.2f} (half of the day's ${total:.2f}).")
print(f"Requests 1 to {number - 1} record ${recorded:.4f}, below the cap. All succeed.")
print(f"Request {number} ({run['extra']['metadata']['ls_model_name']}) starts below the cap and succeeds. It costs ${cost:.4f}, so recorded spend is now ${recorded + cost:.4f}, over the cap.")
print(f"Requests {number + 1} to {len(requests)} ({len(requests) - number} model calls) get a 402 until the window resets at midnight UTC.")

## 6. Which control stops which incident?

Section 1 listed five things that went wrong in the seeded day:

| | Incident |
|---|---|
| A | Release 1.5.0 tells customers with opened electronics that they have 30 days to return them (the policy says 14) |
| B | A burst of 8 errors in `get_order_status` inside 3 minutes |
| C | Orders typed in lower case or without dashes are reported as not found |
| D | Compatibility questions are delegated twice, so they cost about twice as much |
| E | A customer's email address is stored in a trace |

> 💡 **Try it first:** for each incident, pick the control that fits best: an alert, an online evaluator, an Engine issue, a Gateway spend limit, trace masking, or none of them. Then run the next cell.

In [ ]:
answers = [
    ("A", "online evaluator",
     "No governance control prevents a wrong answer. The online evaluator detects it on every trace, and a dataset example of the corrected answer keeps it from coming back."),
    ("B", "alert",
     "An alert with a threshold of 3 failed traces (errored root runs) in 5 minutes fires on the burst and stays quiet for the background errors."),
    ("C", "Engine issue",
     "The tool result has status success, so only a code fix helps. Engine tracks the issue and its pull request."),
    ("D", "Gateway spend limit",
     "A spend limit caps the bill. It does not fix the cause: the duplicate delegation needs a change to the agent."),
    ("E", "trace masking",
     "Client-side masking keeps the address out of the upload. A Gateway data policy scans requests to the provider, not your application's traces."),
]
for incident, control, why in answers:
    print(f"{incident}  {control:<20} {why}")

Two choices are also defensible. For C, the thumbs-down on 16 of the 19 conversations would let an alert on Feedback Score notice the problem; the fix is still a code change. For D, Insights found the cost by subcategory and Engine has it as a finding; the spend limit is the only one of the three that stops the spending.

## 7. Who did what: traces, the audit log, roles and Custom Apps

"Audit logging" covers two different records:

| Question | Where the answer is | Plan |
|---|---|---|
| Who called which model, when, at what cost, and was the call blocked? | The Gateway's traces in the project `gateway`, with `gateway.api_key_id` and `gateway.user_id`, and the Gateway's **Usage** tab | Traces on any plan with the Gateway; **Usage** on Plus or Enterprise |
| Who created, changed or deleted a Gateway policy, a provider secret, an API key or a role? | The organization audit log: **Organization Settings > Audit logs**, kept up to 400 days, with operations such as `create_gateway_policy` | Enterprise; Organization Admin or Organization Operator |
| Which tool did an agent call through the Tool Gateway? | The audit log operations `invoke_mcp_gateway` and `invoke_mcp_server` | Enterprise |

The audit log lists no operation for a single LLM Gateway model call. Model calls are in the Gateway's traces.

Roles decide who may do what. To call models through the Gateway, a key needs `gateway:invoke` and `workspaces:read` in its workspace. To create policies, a person needs `organization:gateway:manage`. On plans other than Enterprise every workspace member is a Workspace Admin; custom roles and access policies are Enterprise features.

A **Custom App** is a UI that runs inside LangSmith and calls the LangSmith API with the permissions of the person viewing it, so each viewer sees only the data they may see. Apps are on the Plus and Enterprise plans, and you build one by chatting with LangSmith or with the `langsmith apps` command line. A review screen for the annotation queue of Section 1 is a typical first app. See the [Custom Apps docs](https://docs.langchain.com/langsmith/custom-apps).

The next cell is the only cell in this module that can spend model tokens. It sends two requests to the Gateway: one to list the models your key may call, and one short chat request. It runs only when you set `GATEWAY_LIVE = True` and a Gateway key is available.

In [ ]:
GATEWAY_LIVE = False  # set to True to call the LLM Gateway; this spends a few model tokens
GATEWAY_BASE_URL = "https://gateway.smith.langchain.com/v1"  # US; the EU host is eu.gateway.smith.langchain.com
gateway_key = os.getenv("LANGSMITH_GATEWAY_API_KEY") or (os.getenv("LANGSMITH_API_KEY") if LANGSMITH_AVAILABLE else None)

if GATEWAY_LIVE and gateway_key:
    import httpx

    headers = {"Authorization": f"Bearer {gateway_key}"}
    try:
        listed = httpx.get(f"{GATEWAY_BASE_URL}/models", headers=headers, timeout=30)
        print("GET /v1/models:", listed.status_code)
        if listed.status_code == 200:
            ids = [model["id"] for model in listed.json().get("data", [])]
            print(f"  {len(ids)} models this key may call, for example: {', '.join(ids[:5])}")
        reply = httpx.post(
            f"{GATEWAY_BASE_URL}/chat/completions",
            headers=headers,
            json={"model": "openai/gpt-5.4-mini", "messages": [{"role": "user", "content": "ping"}]},
            timeout=60,
        )
        print("POST /v1/chat/completions:", reply.status_code)
        print("  Retry-After:", reply.headers.get("Retry-After"))
        print("  X-LangSmith-Gateway-Metadata:", reply.headers.get("X-LangSmith-Gateway-Metadata"))
        print("  Body:", reply.text[:500])
    except httpx.HTTPError as exc:
        print(f"⏭️  The request failed ({type(exc).__name__}). The saved blocks of §5 show the same responses.")
elif GATEWAY_LIVE:
    print("⏭️  Skipped: no Gateway key. Set LANGSMITH_GATEWAY_API_KEY, or a working LANGSMITH_API_KEY, in .env.")
else:
    print("⏭️  Skipped: GATEWAY_LIVE is False. The saved responses of §5 show what a blocked call returns.")

## 8. Where to go next

| Topic | Where |
|---|---|
| Spend, rate, model access and data policies | [Spend](https://docs.langchain.com/langsmith/llm-gateway-spend-policies), [rate limit](https://docs.langchain.com/langsmith/llm-gateway-rate-limit-policies), [model access](https://docs.langchain.com/langsmith/llm-gateway-model-access-policies) and [data](https://docs.langchain.com/langsmith/llm-gateway-data-policy) policies; [header-based policies](https://docs.langchain.com/langsmith/llm-gateway-header-policies) |
| Model fallbacks in the Gateway | [Model fallbacks](https://docs.langchain.com/langsmith/llm-gateway-fallbacks) |
| The Gateway's traces | [View LLM Gateway traces](https://docs.langchain.com/langsmith/llm-gateway-traces) |
| Masking data in traces | [Prevent logging of sensitive data in traces](https://docs.langchain.com/langsmith/mask-inputs-outputs) |
| Audit logs | [Audit logs](https://docs.langchain.com/langsmith/audit-logs) |
| Custom Apps | [Custom Apps](https://docs.langchain.com/langsmith/custom-apps) |
| Engine, end to end | The LangChain Academy course [Autonomous Agent Improvement with LangSmith Engine](https://academy.langchain.com/courses/langsmith-engine) |
| Online evaluation, annotation queues and automation rules on the Module 1 agent | [Module 3, Section 1](../module_3/section_1_production_data_flywheel.ipynb) |
| Offline evaluation of the deep agent, and the Harbor task for the return window | [Module 5, Section 1](../module_5/section_1_baseline_evaluation.ipynb) and [Module 5, Section 5](../module_5/section_5_harbor_evals.ipynb) |

### What you did in code, and where it is in LangSmith

| Step in this notebook | Same step in LangSmith |
|---|---|
| The email search over root runs (§1) | The search bar of your TechHub traces |
| `offline_client(anonymizer=build_anonymizer())` (§3) | Nothing to click: masking is set in the client of your application, before upload |
| `init_chat_model(...)` under five environments (§4) | **LLM Gateway** > **Simulator** shows a request's route |
| `gateway_replay(kind)` (§5) | A blocked request in **LLM Gateway** > **Traces**, with `gateway.request.outcome` set to `blocked` |
| `spend_by(seed, "customer_id")` (§5) | The **Usage** tab with **Breakdown by** **User** or **API key**, or **Custom Dimension** for a header |

## Recap

| Layer | What it blocks | Who configures it |
|---|---|---|
| `PIIMiddleware` | Personal data in what the model sees and in the saved conversation | The developer, in the agent's code |
| Client-side trace masking (`Client(anonymizer=...)`) | Personal data in what LangSmith stores in your traces | The developer, in the client or with environment variables |
| Gateway data policy | Personal data and secrets in requests to the model provider | An organization admin, on the **Data Policy** tab, with the PII entitlement |
| Gateway spend, rate and model access policies | Spend over a cap (`402`), too many requests (`429`), models not on the list (`403`) | An organization admin, with `organization:gateway:manage` |
| Roles and the audit log | Changes by people without the right role; the audit log records who changed what | An organization admin; Enterprise for custom roles and the audit log |

- With `PIIMiddleware` alone, the email was in most uploaded payloads of the trace. An anonymizer brought that to 0 and left the order id alone.
- `LANGSMITH_GATEWAY=true` sends calls to `https://gateway.smith.langchain.com/openai/v1` even when `OPENAI_API_KEY` is set.
- A spend block is a `402` (`APIStatusError`), a rate block a `429` with `Retry-After` (`RateLimitError`), a model access block a `403` (`PermissionDeniedError`). The agent retries only the `429`.
- A spend limit caps the bill after the request that crosses it. It does not fix a costly agent.

This is the last section of Module 7. Across the four sections you detected the day's incidents with online evaluators and alerts, understood them with Insights, fixed them with Engine, and prevented two of them with masking and Gateway policies.